In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

def complete_feature_analysis():
    """
    Análise completa de feature selection para os três modelos
    Modelo 1: Lasso (L1)
    Modelo 2: Ridge (L2) 
    Modelo 3: Ridge (L2)
    """
    
    # Caminhos dos datasets
    model1_train = '/Users/marcelocarvalhoesilva/project/early-obesity-prediction/D-Train-Test Split/MODEL1TRAIN.csv'
    model2_train = '/Users/marcelocarvalhoesilva/project/early-obesity-prediction/D-Train-Test Split/MODEL2TRAIN.csv'
    model3_train = '/Users/marcelocarvalhoesilva/project/early-obesity-prediction/D-Train-Test Split/MODEL3TRAIN.csv'
    
    # Descrições das variáveis em inglês
    variable_descriptions = {
        # Demographics
        'sexo_masculino': 'Male gender',
        'idade_crianca': 'Child age at survey',
        'idade_materna_ao_nascimento': 'Maternal age at birth (years)',
        
        # Geographic regions
        'regiao_Centro-Oeste': 'Central-West region',
        'regiao_Nordeste': 'Northeast region', 
        'regiao_Norte': 'North region',
        'regiao_Sudeste': 'Southeast region',
        'regiao_Sul': 'South region',
        
        # Race/ethnicity
        'cor_Branca': 'White race/ethnicity',
        'cor_Outros': 'Other race/ethnicity',
        'cor_Parda (mulata, cabocla, cafuza, mameluca ou mestiça)': 'Mixed race/ethnicity',
        'cor_Preta': 'Black race/ethnicity',
        
        # Perinatal factors
        'semanas_gravidez': 'Gestational age at birth (weeks)',
        'peso_nascimento': 'Birth weight (grams)',
        'altura_nascimento': 'Birth length (cm)',
        
        # Delivery type
        'parto_Cesariana agendada (eletiva)': 'Elective cesarean delivery',
        'parto_Cesariana de urgência (Não agendada)': 'Emergency cesarean delivery',
        'parto_Normal': 'Vaginal delivery',
        'tipo_de_parto': 'Delivery type (binary)',
        
        # Early practices and health
        'chupeta_usou': 'Pacifier use',
        'sabe_ler': 'Maternal literacy',
        'sindrome_nao': 'No genetic syndromes',
        'teve_exposicao_escola': 'School exposure',
        
        # Socioeconomic
        'nivel_educacional_mae': 'Maternal education level',
        'k05_prenatal_consultas': 'Number of prenatal visits',
        'recebe_beneficio': 'Receives government benefits',
        
        # Maternal weight/health
        'k06_peso_engravidar': 'Pre-pregnancy weight (kg)',
        'k07_peso_final': 'Final pregnancy weight (kg)', 
        'k08_quilos': 'Gestational weight gain (kg)',
        'peso_materno': 'Current maternal weight (kg)',
        'vd_ien_escore': 'Food insecurity score',
        
        # Prenatal care
        'k03_prenatal': 'Received prenatal care',
        'k04_prenatal_semanas': 'Gestational age at prenatal care initiation (weeks)',
        
        # Breastfeeding practices
        'k11_amamentou': 'Ever breastfed',
        'tempo_primeira_mamada_horas': 'Time to first breastfeed (hours)',
        'usou_utensilios_amamentacao': 'Used breastfeeding equipment',
        'dias_aleitamento_exclusivo': 'Exclusive breastfeeding duration (days)',
        
        # Milk banking and support
        'doou_leite_banco': 'Donated breast milk to milk bank',
        'recebeu_leite_banco': 'Received milk from milk bank',
        'amamentou_outra_crianca': 'Breastfed another child',
        
        # Feeding behaviors
        'usou_mamadeira': 'Bottle feeding use',
        'deixou_amamentar_por_outra': 'Let another person breastfeed child',
        'busca_info_aleitamento': 'Sought breastfeeding information',
        'deu_outros_liquidos': 'Gave other liquids besides breast milk',
        'k15_recebeu': 'Received feeding support'
    }
    
    def analyze_model(train_path, model_name, regularization_type, C_value):
        """
        Analisa um modelo específico extraindo coeficientes significativos
        """
        print(f"\n{'='*80}")
        print(f"ANALYZING {model_name}")
        print(f"{'='*80}")
        
        # Carregar dados
        df_train = pd.read_csv(train_path)
        
        # Corrigir tipos de dados para variáveis binárias (modelos 2 e 3)
        binary_vars = [
            'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca',
            'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento',
            'deu_outros_liquidos', 'k15_recebeu', 'k11_amamentou', 'k03_prenatal',
            'usou_utensilios_amamentacao'
        ]
        
        for var in binary_vars:
            if var in df_train.columns:
                df_train[var] = df_train[var].astype(int)
        
        # Preparar dados
        target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
        y_train = df_train['obeso'].copy()
        X_train = df_train.drop(target_cols + ['id_anon'], axis=1)
        
        print(f"Dataset {model_name}:")
        print(f"  - Observations: {len(df_train):,}")
        print(f"  - Features: {X_train.shape[1]}")
        print(f"  - Obese: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
        
        # Configurar modelo baseado na regularização ótima
        if regularization_type == 'lasso':
            penalty = 'l1'
            solver = 'liblinear'
        else:  # ridge
            penalty = 'l2'
            solver = 'lbfgs'
        
        model = Pipeline([
            ('scaler', RobustScaler()),
            ('logistic', LogisticRegression(
                random_state=42,
                max_iter=10000,
                penalty=penalty,
                C=C_value,
                class_weight='balanced',
                solver=solver
            ))
        ])
        
        print(f"Model configuration:")
        print(f"  - Algorithm: Logistic Regression")
        print(f"  - Regularization: {regularization_type.upper()}")
        print(f"  - C parameter: {C_value}")
        
        # Treinar modelo
        model.fit(X_train, y_train)
        
        # Extrair coeficientes
        feature_names = X_train.columns.tolist()
        coefficients = model.named_steps['logistic'].coef_[0]
        
        # Bootstrap para intervalos de confiança
        print(f"Calculating bootstrap confidence intervals (n=1,000)...")
        
        n_bootstrap = 1000
        np.random.seed(42)
        bootstrap_coefficients = []
        
        for i in range(n_bootstrap):
            # Amostragem bootstrap
            indices = np.random.choice(len(y_train), size=len(y_train), replace=True)
            X_boot = X_train.iloc[indices]
            y_boot = y_train.iloc[indices]
            
            try:
                model_boot = Pipeline([
                    ('scaler', RobustScaler()),
                    ('logistic', LogisticRegression(
                        random_state=42,
                        max_iter=10000,
                        penalty=penalty,
                        C=C_value,
                        class_weight='balanced',
                        solver=solver
                    ))
                ])
                
                model_boot.fit(X_boot, y_boot)
                coeffs_boot = model_boot.named_steps['logistic'].coef_[0]
                bootstrap_coefficients.append(coeffs_boot)
                
            except Exception:
                bootstrap_coefficients.append(coefficients)
        
        bootstrap_coefficients = np.array(bootstrap_coefficients)
        
        # Calcular intervalos de confiança 95%
        ci_lower = np.percentile(bootstrap_coefficients, 2.5, axis=0)
        ci_upper = np.percentile(bootstrap_coefficients, 97.5, axis=0)
        
        # Criar DataFrame com resultados
        results_df = pd.DataFrame({
            'variable': feature_names,
            'description': [variable_descriptions.get(var, var) for var in feature_names],
            'coefficient': coefficients,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper
        })
        
        # Adicionar análises
        results_df['ci_95'] = results_df.apply(lambda x: f"[{x['ci_lower']:.3f}, {x['ci_upper']:.3f}]", axis=1)
        results_df['significant'] = (results_df['ci_lower'] > 0) | (results_df['ci_upper'] < 0)
        # FIX: In LogisticRegression, positive coefficient = increased log-odds of obese=1 = Risk
        #      Negative coefficient = decreased log-odds of obese=1 = Protective
        results_df['effect_direction'] = results_df['coefficient'].apply(
            lambda x: 'Risk' if x > 0 else 'Protective'
        )
        results_df['abs_coefficient'] = np.abs(results_df['coefficient'])
        
        # Filtrar apenas features significativas
        significant_features = results_df[results_df['significant']].copy()
        significant_features = significant_features.sort_values('abs_coefficient', ascending=False).reset_index(drop=True)
        significant_features['rank'] = range(1, len(significant_features) + 1)
        
        # Estatísticas
        n_total = len(results_df)
        n_significant = len(significant_features)
        n_risk = len(significant_features[significant_features['coefficient'] > 0])
        n_protective = len(significant_features[significant_features['coefficient'] < 0])
        
        print(f"\nResults summary:")
        print(f"  - Total features: {n_total}")
        print(f"  - Statistically significant: {n_significant} ({n_significant/n_total*100:.1f}%)")
        print(f"  - Risk effects: {n_risk}")
        print(f"  - Protective effects: {n_protective}")
        
        if len(significant_features) > 0:
            print(f"\nStatistically significant features:")
            for _, row in significant_features.iterrows():
                print(f"  {row['rank']}. {row['variable']}: {row['coefficient']:.3f} {row['ci_95']} ({row['effect_direction']})")
        else:
            print(f"\n❌ NO STATISTICALLY SIGNIFICANT FEATURES")
        
        return significant_features, results_df
    
    # Analisar cada modelo
    print("COMPLETE FEATURE ANALYSIS - THREE MODELS")
    print("="*80)
    
    # Modelo 1: Lasso
    model1_sig, model1_all = analyze_model(
        model1_train, 
        "MODEL 1 (DEMOGRAPHIC/PERINATAL)", 
        "lasso", 
        100.0  # C value típico para Lasso
    )
    
    # Modelo 2: Ridge  
    model2_sig, model2_all = analyze_model(
        model2_train, 
        "MODEL 2 (FEEDING PRACTICES)", 
        "ridge", 
        100.0  # C value típico para Ridge
    )
    
    # Modelo 3: Ridge
    model3_sig, model3_all = analyze_model(
        model3_train, 
        "MODEL 3 (COMBINED)", 
        "ridge", 
        100.0  # C value típico para Ridge
    )
    
    # Criar tabela consolidada para o paper
    print(f"\n{'='*100}")
    print("CONSOLIDATED SIGNIFICANT FEATURES TABLE FOR MANUSCRIPT")
    print("="*100)
    
    # Adicionar identificador do modelo
    model1_sig['model'] = 'Model 1'
    model2_sig['model'] = 'Model 2' 
    model3_sig['model'] = 'Model 3'
    
    # Combinar todos os resultados significativos
    all_significant = pd.concat([model1_sig, model2_sig, model3_sig], ignore_index=True)
    
    if len(all_significant) > 0:
        print(f"\nTable: Statistically Significant Feature Associations with Childhood Obesity")
        print(f"{'='*100}")
        print(f"{'Model':<8} {'Rank':<4} {'Variable':<35} {'Description':<45} {'Coeff [95% CI]':<25} {'Effect'}")
        print("-" * 120)
        
        for _, row in all_significant.iterrows():
            coeff_ci = f"{row['coefficient']:.3f} {row['ci_95']}"
            print(f"{row['model']:<8} {row['rank']:<4} {row['variable']:<35} {row['description']:<45} {coeff_ci:<25} {row['effect_direction']}")
    else:
        print(f"\n❌ NO STATISTICALLY SIGNIFICANT FEATURES ACROSS ALL MODELS")
        print("This provides strong evidence that early-life factors lack robust associations with childhood obesity")
    
    # Resumo geral
    print(f"\n{'='*80}")
    print("GENERAL SUMMARY")
    print("="*80)
    print(f"Model 1 significant features: {len(model1_sig)}/{len(model1_all)} ({len(model1_sig)/len(model1_all)*100:.1f}%)")
    print(f"Model 2 significant features: {len(model2_sig)}/{len(model2_all)} ({len(model2_sig)/len(model2_all)*100:.1f}%)")
    print(f"Model 3 significant features: {len(model3_sig)}/{len(model3_all)} ({len(model3_sig)/len(model3_all)*100:.1f}%)")
    print(f"Total significant across all models: {len(all_significant)}")
    
    # Salvar resultados
    all_significant.to_csv('all_models_significant_features.csv', index=False)
    print(f"\n📊 Significant features saved to: all_models_significant_features.csv")
    
    return model1_sig, model2_sig, model3_sig, all_significant

# Executar análise completa
if __name__ == "__main__":
    model1_results, model2_results, model3_results, consolidated_results = complete_feature_analysis()